# Dynamic Factor Model for Nowcasting

This notebook demonstrates how to use **Dynamic Factor Models (DFM)** for nowcasting GDP growth
using mixed-frequency economic indicators.

**Reference**: Giannone, D., Reichlin, L., & Small, D. (2008). "Nowcasting: The real-time informational
content of macroeconomic data." *Journal of Monetary Economics*, 55(4), 665-676.

The DFM approach extracts common latent factors from a large panel of indicators observed at different
frequencies, using a state-space representation estimated via the **EM algorithm** with
**Kalman filter/smoother**.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.nowcasting import DFMNowcaster, NewsDecomposition

# Add helpers path
sys.path.insert(0, "../utils")
from helpers import load_mixed_freq, load_macro_brazil, simulate_ragged_edge

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. The Nowcasting Problem

Nowcasting is the prediction of the **present** or very near future. GDP is released with a
significant delay (typically 1-3 months after the quarter ends), but monthly indicators such as
industrial production, retail sales, and confidence indices are available much sooner.

The key challenge is the **ragged edge**: at any point in time, different indicators have
different amounts of data available, creating an unbalanced panel with missing observations
at the end of each series.

In [ ]:
# Load the mixed-frequency dataset
data = load_mixed_freq()
print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nMissing values per column:")
print(data.isna().sum())

# Visualize the ragged edge pattern
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flat, data.columns):
    series = data[col].dropna()
    ax.plot(series.index, series.values, "b-", linewidth=1.2)
    ax.set_title(col.replace("_", " ").title(), fontsize=12)
    ax.grid(True, alpha=0.3)
    # Mark NaN periods
    nan_mask = data[col].isna()
    if nan_mask.any():
        for idx in data.index[nan_mask]:
            ax.axvline(idx, color="red", alpha=0.1, linewidth=0.5)

fig.suptitle("Mixed-Frequency Data with Ragged Edge", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Show the ragged edge pattern explicitly
print("\nRagged Edge Pattern (last 6 months):")
print(data.tail(6).to_string())

## 2. Dynamic Factor Model

The DFM represents the co-movement of a panel of $N$ indicators $x_t = (x_{1t}, \ldots, x_{Nt})'$
through a small number of latent factors $f_t$:

**Observation equation (measurement):**
$$x_t = \Lambda f_t + e_t, \quad e_t \sim N(0, R)$$

**State equation (transition):**
$$f_t = A_1 f_{t-1} + A_2 f_{t-2} + \ldots + A_p f_{t-p} + u_t, \quad u_t \sim N(0, Q)$$

Where:
- $\Lambda$ is the matrix of **factor loadings** (how each indicator loads on the factors)
- $A_1, \ldots, A_p$ are the **VAR coefficients** for factor dynamics
- $R$ is the **observation noise** covariance (diagonal, idiosyncratic)
- $Q$ is the **state noise** covariance

The **Kalman filter** handles missing data naturally: when an observation is missing,
the filter skips the update step for that variable, using only the prediction.

For **mixed-frequency** data, quarterly variables use the **Mariano-Murasawa (2003)**
triangular accumulator in the state space, linking the quarterly flow to monthly factors.

In [ ]:
# Define frequency map: which variables are monthly vs quarterly
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

# Create and fit DFM with 1 factor
dfm = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
    em_tol=1e-6,
)

dfm.fit(data)
print(dfm)

# Show factor loadings
print("\nFactor Loadings:")
print(dfm.loadings())

## 3. Extracting Factors

The latent factor captures the common dynamics across all indicators. A positive loading
means the indicator moves with the factor; a negative loading means it moves against.

We can visualize the estimated factor against the observed GDP growth to see how well
the factor tracks the target variable.

In [ ]:
# Extract estimated factors
factors = dfm.factors()
print(f"Factors shape: {factors.shape}")
print(f"\nFactor statistics:")
print(factors.describe())

# Plot factor vs GDP growth
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot factor on left axis
color1 = "steelblue"
ax1.plot(factors.index, factors["factor_1"], color=color1, linewidth=2, label="Latent Factor 1")
ax1.set_xlabel("Date")
ax1.set_ylabel("Factor Value", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

# Plot GDP on right axis
ax2 = ax1.twinx()
color2 = "darkorange"
gdp_obs = data["gdp_growth"].dropna()
ax2.scatter(gdp_obs.index, gdp_obs.values, color=color2, s=40, zorder=5, label="GDP Growth (quarterly)")
ax2.set_ylabel("GDP Growth", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

ax1.set_title("Estimated Latent Factor vs GDP Growth", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Nowcasting GDP

The key advantage of the DFM approach is its ability to generate a nowcast even when the
data panel has a **ragged edge** — i.e., different indicators have data available up to
different points in time.

We simulate this by removing the last few observations from some indicators.

In [ ]:
# Simulate ragged edge: some indicators missing more recent data
ragged_data = simulate_ragged_edge(data, {
    "industrial_production": 1,   # 1 month missing
    "retail_sales": 2,            # 2 months missing
    "confidence_index": 0,        # fully available
    "gdp_growth": 3,              # 3 months missing (typical for GDP)
})

print("Ragged edge pattern (last 6 months):")
print(ragged_data.tail(6).to_string())

# Fit DFM on ragged-edge data and nowcast
dfm_ragged = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
)
dfm_ragged.fit(ragged_data)

nowcast = dfm_ragged.nowcast(target="gdp_growth")
print(f"\nGDP Nowcast: {nowcast.point[0]:.4f}")
print(f"80% CI: [{nowcast.lower_80[0]:.4f}, {nowcast.upper_80[0]:.4f}]")
print(f"95% CI: [{nowcast.lower_95[0]:.4f}, {nowcast.upper_95[0]:.4f}]")
print(f"Model: {nowcast.model_name}")

## 5. News Decomposition

A powerful feature of the DFM framework is the ability to decompose nowcast **revisions**
into contributions from each newly released data point. This follows
**Banbura & Modugno (2014)**.

When new data arrives, the nowcast revision is:

$$\Delta \hat{y}_{t|\Omega_{new}} - \hat{y}_{t|\Omega_{old}} = \sum_i w_i \cdot (x_i^{new} - E[x_i | \Omega_{old}])$$

Where:
- $w_i$ are the **weights** (sensitivity of the nowcast to indicator $i$)
- $x_i^{new} - E[x_i | \Omega_{old}]$ is the **news** (surprise in the new release)

In [ ]:
# Create old and new information sets to demonstrate news decomposition
# Old: more missing data
old_data = simulate_ragged_edge(data, {
    "industrial_production": 3,
    "retail_sales": 3,
    "confidence_index": 2,
    "gdp_growth": 4,
})

# New: some indicators updated
new_data = simulate_ragged_edge(data, {
    "industrial_production": 1,
    "retail_sales": 2,
    "confidence_index": 0,
    "gdp_growth": 4,
})

# Perform news decomposition
news_decomp = NewsDecomposition(dfm)
result = news_decomp.decompose(old_data, new_data, target="gdp_growth")

# Print summary
print(result.summary())

# Plot contributions as bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

result.plot_contributions(ax=axes[0])
axes[0].set_title("News Contributions to Nowcast Revision", fontsize=12)

result.plot_waterfall(ax=axes[1])
axes[1].set_title("Waterfall: Old Nowcast → New Nowcast", fontsize=12)

plt.tight_layout()
plt.show()

## 6. Pseudo Real-Time Exercise

To evaluate nowcasting performance, we simulate a **pseudo real-time** exercise:
at each point in time, we only use data that would have been available, then generate
a nowcast and compare it to the actual GDP release.

This mimics how the model would perform in practice, accounting for the ragged edge.

In [ ]:
# Pseudo real-time exercise: simulate sequential data arrival
gdp_dates = data["gdp_growth"].dropna().index
# Use the last 8 quarters for evaluation
eval_dates = gdp_dates[-8:]

nowcast_results = []

for eval_date in eval_dates:
    # Simulate data available at different lags within the quarter
    for months_before_release in [3, 2, 1]:
        # Create ragged-edge data as if we were `months_before_release` months
        # before the GDP release
        cutoff_idx = data.index.get_loc(eval_date) - months_before_release
        if cutoff_idx < 12:
            continue

        available_data = data.iloc[:cutoff_idx + 1].copy()
        # GDP is not yet available for this quarter
        available_data.loc[eval_date:, "gdp_growth"] = np.nan

        # Fit and nowcast
        rt_dfm = DFMNowcaster(
            n_factors=1,
            factor_lags=2,
            frequency_map=frequency_map,
            aggregation="sum",
            em_iterations=50,
        )
        try:
            rt_dfm.fit(available_data)
            fc = rt_dfm.nowcast(target="gdp_growth")
            actual = data.loc[eval_date, "gdp_growth"]

            nowcast_results.append({
                "quarter": eval_date,
                "months_before": months_before_release,
                "nowcast": fc.point[0],
                "actual": actual,
                "error": fc.point[0] - actual,
            })
        except Exception:
            pass

results_df = pd.DataFrame(nowcast_results)
print("Pseudo Real-Time Results:")
print(results_df.to_string(index=False))

# Plot nowcast evolution for each quarter
fig, ax = plt.subplots(figsize=(14, 6))

for quarter in results_df["quarter"].unique():
    qdata = results_df[results_df["quarter"] == quarter].sort_values("months_before", ascending=False)
    label = quarter.strftime("%Y-Q%q") if hasattr(quarter, "strftime") else str(quarter)
    ax.plot(qdata["months_before"], qdata["nowcast"], "o-", label=f"{quarter.year}Q{(quarter.month-1)//3+1}")
    ax.axhline(qdata["actual"].iloc[0], linestyle="--", alpha=0.3)

ax.set_xlabel("Months Before GDP Release")
ax.set_ylabel("Nowcast Value")
ax.set_title("Nowcast Evolution as Data Arrives", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
ax.invert_xaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# RMSE by horizon
print("\nRMSE by months before release:")
for m in sorted(results_df["months_before"].unique()):
    subset = results_df[results_df["months_before"] == m]
    rmse = np.sqrt(np.mean(subset["error"] ** 2))
    print(f"  {m} months before: RMSE = {rmse:.4f}")

### Exercise 1: DFM with 2 factors for Brazilian macro

Load the `macro_brazil.csv` dataset and fit a DFM with 2 factors. Compare the factors
to GDP growth and inflation. Which indicators load most strongly on each factor?

In [ ]:
# TODO: Exercise 1
# Hints:
# 1. Load macro_brazil with load_macro_brazil()
# 2. Define a frequency_map (gdp_growth is quarterly, others are monthly)
# 3. Create DFMNowcaster(n_factors=2, ...)
# 4. Examine loadings() to see which indicators load on each factor
# 5. Plot both factors vs GDP growth

### Exercise 2: Compare DFM with different numbers of factors

Fit DFM models with 1, 2, and 3 factors on the `mixed_freq.csv` dataset.
Compare their nowcast accuracy using a pseudo real-time exercise.
Does adding more factors improve the nowcast?

In [ ]:
# TODO: Exercise 2
# Hints:
# 1. Loop over n_factors in [1, 2, 3]
# 2. For each, run the pseudo real-time exercise from Section 6
# 3. Compute RMSE for each model
# 4. Plot RMSE vs number of factors
# 5. Discuss the bias-variance tradeoff in factor selection